# First-layer ESD: binning diagnostics and direct `powerlaw` MLE fits

This notebook extends `05_muonclip_esd_clip_xmax.ipynb` but analyzes only the first WeightWatcher matrix (`L00_W_Q`). WeightWatcher supplies the raw ESD and reference standard/`clip_xmax` fits; all plots and direct fits below are constructed independently. Set `RUN_DIR` before launch. Outputs go to `$RUN_DIR/diagnostics/first_layer_esd_powerlaw_step_XXXXXXX/`.

The notebook tests log/linear/quantile binning, colors bins by exact occupancy, plots an unbinned CCDF, and fits the full ESD and an automatically selected tail with the Python `powerlaw` package while removing top eigenvalues one at a time.


In [ ]:
from pathlib import Path
import json, math, os, random, sys, warnings
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, LogNorm
import numpy as np
import pandas as pd
import powerlaw
import torch
from IPython.display import display
import weightwatcher as ww

cwd=Path.cwd().resolve()
roots=[cwd,cwd.parent,cwd/"baseline"/"nanogpt_one_head"]
ROOT=next((p for p in roots if (p/"configs"/"reference.yaml").is_file()),None)
if ROOT is None: raise FileNotFoundError("Run from the repo root, baseline/nanogpt_one_head, or its notebooks directory")
sys.path.insert(0,str(ROOT/"src"))
from rg_nanogpt_one_head.model import GPT,GPTConfig
from rg_nanogpt_one_head.spectral import WeightMatrixHolder,_attach_matrix_metadata

RUN_DIR=Path(os.environ.get("RUN_DIR","")).expanduser().resolve()
if not os.environ.get("RUN_DIR","").strip(): raise EnvironmentError("Set RUN_DIR before launching Jupyter")
TARGET_EPOCH=None
MIN_EVALS=20
MAX_FINGERS=10
MAX_TOP_TRIM=20
MIN_RETAINED=20
LOG_BINS=(16,32,64,128)
REF_BINS=64
FIT_BOUNDED_SUPPORT=True
print("ROOT",ROOT,"RUN_DIR",RUN_DIR)
print("WeightWatcher",getattr(ww,"__version__","unknown"),"powerlaw",getattr(powerlaw,"__version__","unknown"))

## Load a checkpoint and extract the first ESD

`TARGET_EPOCH=None` selects the latest row in `epoch_metrics.csv`. WeightWatcher is run with `plot=False` for the standard fit and for `fix_fingers='clip_xmax'`; the notebook then retrieves the first layer's raw spectrum with `get_ESD`.


In [ ]:
if not RUN_DIR.is_dir(): raise FileNotFoundError(RUN_DIR)
manifest=json.loads((RUN_DIR/"manifest.json").read_text())
metrics=pd.read_csv(RUN_DIR/"epoch_metrics.csv")
for c in ("step","nominal_epoch","epoch"):
    if c in metrics: metrics[c]=pd.to_numeric(metrics[c],errors="coerce")
metrics=metrics.dropna(subset=["step","nominal_epoch"]).sort_values("step")
row=metrics.iloc[-1] if TARGET_EPOCH is None else metrics.loc[(metrics.nominal_epoch-float(TARGET_EPOCH)).abs().idxmin()]
STEP=int(row.step); NOMINAL_EPOCH=float(row.nominal_epoch)
checkpoint=Path(str(row.get("checkpoint_path","")))
if not checkpoint.is_file():
    matches=sorted((RUN_DIR/"epoch_checkpoints").glob(f"*step_{STEP:07d}.pt"))
    if len(matches)!=1: raise FileNotFoundError(f"checkpoint for step {STEP}: {matches}")
    checkpoint=matches[0]
payload=torch.load(checkpoint,map_location="cpu",weights_only=False)
model=GPT(GPTConfig(**manifest["model"])); model.load_state_dict(payload["model"]); model.eval()
holder=WeightMatrixHolder(model)
out=RUN_DIR/"diagnostics"/f"first_layer_esd_powerlaw_step_{STEP:07d}"; out.mkdir(parents=True,exist_ok=True)
seed=int(manifest["seed"])+1_000_003+STEP
def reset_seed():
    random.seed(seed); np.random.seed(seed%(2**32-1)); torch.manual_seed(seed)
def attach(x): return _attach_matrix_metadata(pd.DataFrame(x),holder.matrix_metadata)
reset_seed(); ws=ww.WeightWatcher(model=holder)
std=attach(ws.analyze(ERG=False,randomize=False,plot=False,min_evals=MIN_EVALS))
reset_seed(); wc=ww.WeightWatcher(model=holder)
clip=attach(wc.analyze(ERG=False,randomize=False,plot=False,min_evals=MIN_EVALS,fix_fingers="clip_xmax",max_fingers=MAX_FINGERS))
first=std.sort_values(["layer_id","matrix_name"]).iloc[0]
name=str(first.matrix_name); layer_id=int(first.layer_id)
if name!="L00_W_Q": raise RuntimeError(f"Expected L00_W_Q, got {name}")
first_clip=clip.loc[clip.matrix_name==name].iloc[0]
esd=np.asarray(ws.get_ESD(layer=layer_id),dtype=float).reshape(-1)
esd=np.sort(esd[np.isfinite(esd)&(esd>0)])
if esd.size<MIN_RETAINED: raise RuntimeError(f"Only {esd.size} positive eigenvalues")
pd.DataFrame({"rank":np.arange(1,esd.size+1),"eigenvalue":esd}).to_csv(out/"first_layer_esd.csv",index=False)
cols=[c for c in ("matrix_name","layer_id","alpha","sigma","D","xmin","xmax","num_fingers","warning") if c in std.columns or c in clip.columns]
refs=pd.concat([pd.DataFrame([first]).assign(method="weightwatcher_standard"),pd.DataFrame([first_clip]).assign(method="weightwatcher_clip_xmax")])
refs[["method"]+[c for c in cols if c in refs]].to_csv(out/"weightwatcher_reference_fits.csv",index=False)
display(refs[["method"]+[c for c in cols if c in refs]])
print(checkpoint,name,"n=",esd.size,"range=",(esd.min(),esd.max()),"output=",out)

## Binning diagnostics

For density-normalized bins, $\widehat\rho_i=n_i/(N\Delta\lambda_i)$. Geometric bins have $\Delta\lambda_i\propto\lambda_i$, so fixed occupancies create apparent $\lambda^{-1}$ ridges. Coloring each point by `n_i` and plotting $\lambda_i\widehat\rho_i$ makes that quantization explicit.


In [ ]:
def clean(x):
    x=np.asarray(x,float).reshape(-1); return np.sort(x[np.isfinite(x)&(x>0)])
def edges(x,n,kind):
    x=clean(x); lo=max(np.nextafter(x.min(),0),np.finfo(float).tiny); hi=np.nextafter(x.max(),np.inf)
    if kind=="log": return np.geomspace(lo,hi,n+1)
    if kind=="linear": return np.linspace(lo,hi,n+1)
    e=np.unique(np.quantile(x,np.linspace(0,1,n+1))); e[0]=lo; e[-1]=hi; return e
def hist(x,e,label):
    x=clean(x); c,e=np.histogram(x,bins=e); w=np.diff(e); z=np.sqrt(e[:-1]*e[1:])
    return pd.DataFrame({"scheme":label,"left":e[:-1],"right":e[1:],"center":z,"width":w,"count":c,"density":c/(len(x)*w),"lambda_density":z*c/(len(x)*w)})
def scatter(ax,h,title,log_color=False,guides=None):
    q=h[(h["count"]>0)&(h["density"]>0)]; m=int(q["count"].max())
    norm=LogNorm(1,m) if log_color and m>1 else BoundaryNorm(np.arange(.5,m+1.5),plt.get_cmap("viridis").N)
    sc=ax.scatter(q.center,q.density,c=q["count"],cmap="viridis",norm=norm,s=28+9*np.sqrt(q["count"]),edgecolors="none")
    ax.set(xscale="log",yscale="log",xlabel=r"$\lambda$",ylabel=r"$\widehat\rho(\lambda)$",title=title); ax.grid(True,which="both",alpha=.2)
    plt.colorbar(sc,ax=ax,pad=.01,label="eigenvalues/bin")
    if guides is not None:
        e=guides; r=e[1]/e[0]; f=math.sqrt(r)-1/math.sqrt(r); g=np.geomspace(e[0],e[-1],300)
        for k in (1,2): ax.plot(g,k/(len(esd)*f*g),"--",lw=1,label=f"exact {k}/bin ridge")
        ax.legend(fontsize=8)

fig,axs=plt.subplots(2,2,figsize=(14,10),constrained_layout=True)
log_frames={}
for ax,n in zip(axs.ravel(),LOG_BINS):
    e=edges(esd,n,"log"); h=hist(esd,e,f"log_{n}"); log_frames[n]=h; h.to_csv(out/f"hist_log_{n}.csv",index=False)
    scatter(ax,h,f"{n} logarithmic bins",guides=e)
fig.suptitle(f"{name}: log-bin resolution and exact occupancy"); fig.savefig(out/"log_bin_resolution.png",dpi=180); plt.show()

families=[("log",edges(esd,64,"log")),("linear",edges(esd,64,"linear")),("quantile",edges(esd,32,"quantile"))]
fig,axs=plt.subplots(1,3,figsize=(17,5),constrained_layout=True)
for ax,(label,e) in zip(axs,families):
    h=hist(esd,e,label); h.to_csv(out/f"hist_{label}.csv",index=False); scatter(ax,h,label,log_color=True)
fig.suptitle(f"{name}: bin-structure sensitivity"); fig.savefig(out/"bin_structure_comparison.png",dpi=180); plt.show()

h=log_frames[REF_BINS]; q=h[h["count"]>0].copy(); e=edges(esd,REF_BINS,"log"); r=e[1]/e[0]; f=math.sqrt(r)-1/math.sqrt(r)
q["expected"]=q["count"]/(len(esd)*f)
summary=q.groupby("count").agg(n_bins=("count","size"),median_lambda_density=("lambda_density","median"),expected=("expected","first")).reset_index()
summary.to_csv(out/"occupancy_levels.csv",index=False); display(summary)
fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True); m=int(q["count"].max())
sc=ax.scatter(q.center,q.lambda_density,c=q["count"],cmap="viridis",norm=BoundaryNorm(np.arange(.5,m+1.5),plt.get_cmap("viridis").N),s=35+9*np.sqrt(q["count"]),edgecolors="none")
for k in sorted(q["count"].unique()): ax.axhline(k/(len(esd)*f),ls="--",lw=.8,alpha=.6)
ax.set(xscale="log",yscale="log",xlabel=r"$\lambda$",ylabel=r"$\lambda\widehat\rho(\lambda)$",title=f"{name}: occupancy-coordinate collapse"); ax.grid(True,which="both",alpha=.2); plt.colorbar(sc,ax=ax,label="eigenvalues/bin")
fig.savefig(out/"occupancy_coordinate.png",dpi=180); plt.show()

## Unbinned CCDF and direct MLE trimming sweep

The CCDF does not depend on histogram bins. For each `k`, the notebook removes the `k` largest eigenvalues, then fits (a) every retained value by fixing `xmin` to the retained minimum and (b) an auto-selected tail. The usual unbounded power law is primary; an optional bounded-support fit makes the upper-clipping assumption explicit.


In [ ]:
x=np.sort(esd); ccdf=(len(x)-np.arange(len(x)))/len(x)
fig,ax=plt.subplots(figsize=(8,5),constrained_layout=True); ax.loglog(x,ccdf,"."); ax.set(xlabel=r"$\lambda$",ylabel=r"$P(\Lambda\geq\lambda)$",title=f"{name}: empirical unbinned CCDF"); ax.grid(True,which="both",alpha=.2); fig.savefig(out/"empirical_ccdf.png",dpi=180); plt.show()

def fnum(v):
    try: z=float(v)
    except (TypeError,ValueError): return np.nan
    return z if np.isfinite(z) else np.nan
def fit_one(data,k,scope,support):
    kw={"discrete":False,"verbose":False}
    if scope=="full": kw["xmin"]=float(data.min())
    if support=="bounded": kw["xmax"]=float(data.max())
    base={"trim_top_k":k,"scope":scope,"support":support,"n_retained":len(data),"retained_max":data.max(),"alpha":np.nan,"sigma":np.nan,"D":np.nan,"xmin":np.nan,"xmax_model":np.nan,"n_tail":0,"tail_fraction":np.nan,"error":""}
    try:
        with warnings.catch_warnings(): warnings.simplefilter("ignore"); f=powerlaw.Fit(data,**kw); p=f.power_law
        xmin=fnum(getattr(p,"xmin",getattr(f,"xmin",np.nan))); nt=int((data>=xmin).sum()) if np.isfinite(xmin) else 0
        base.update(alpha=fnum(getattr(p,"alpha",np.nan)),sigma=fnum(getattr(p,"sigma",np.nan)),D=fnum(getattr(p,"D",np.nan)),xmin=xmin,xmax_model=fnum(getattr(p,"xmax",getattr(f,"xmax",np.nan))),n_tail=nt,tail_fraction=nt/len(data))
    except Exception as exc: base["error"]=f"{type(exc).__name__}: {exc}"
    return base

rows=[]; max_trim=min(MAX_TOP_TRIM,len(esd)-MIN_RETAINED); supports=["unbounded"]+(["bounded"] if FIT_BOUNDED_SUPPORT else [])
for k in range(max_trim+1):
    data=esd if k==0 else esd[:-k]
    for scope in ("full","tail"):
        for support in supports: rows.append(fit_one(data,k,scope,support))
sweep=pd.DataFrame(rows); sweep.to_csv(out/"powerlaw_mle_trim_sweep.csv",index=False)
ok=sweep[sweep.error.eq("")].copy(); display(ok.head(12))
fig,axs=plt.subplots(2,2,figsize=(14,9),constrained_layout=True)
for ax,(metric,label,logy) in zip(axs.ravel(),[("alpha",r"$\alpha$",False),("D","KS D",False),("xmin",r"$x_{min}$",True),("n_tail","tail size",False)]):
    for (scope,support),g in ok.groupby(["scope","support"]): ax.plot(g.trim_top_k,g[metric],"o-",ms=3,label=f"{scope}, {support}")
    ax.set(xlabel="largest eigenvalues removed",ylabel=label,title=label); ax.grid(True,which="both",alpha=.2)
    if logy: ax.set_yscale("log")
    ax.legend(fontsize=8)
fig.suptitle(f"{name}: direct powerlaw MLE under one-at-a-time top trimming"); fig.savefig(out/"powerlaw_trim_sweep.png",dpi=180); plt.show()

primary=ok[ok.support.eq("unbounded")]
ww_k=fnum(first_clip.get("num_fingers",np.nan)); candidates={0}
if np.isfinite(ww_k) and int(ww_k)<=max_trim: candidates.add(int(ww_k))
for scope in ("full","tail"):
    g=primary[(primary.scope==scope)&(primary.n_tail>=MIN_RETAINED)]
    if len(g): candidates.add(int(g.sort_values(["D","trim_top_k"]).iloc[0].trim_top_k))
print("candidate trim counts",sorted(candidates))

fig,ax=plt.subplots(figsize=(9,6),constrained_layout=True); ax.loglog(x,ccdf,".",label="empirical CCDF")
grid=np.geomspace(x.min(),x.max(),700)
for _,r in primary[primary.trim_top_k.isin(candidates)].sort_values(["trim_top_k","scope"]).iterrows():
    valid=grid>=r.xmin; y=np.full_like(grid,np.nan); y[valid]=r.tail_fraction*(grid[valid]/r.xmin)**(1-r.alpha)
    ax.loglog(grid,y,lw=1.3,label=f"k={int(r.trim_top_k)} {r.scope}: a={r.alpha:.3f}, D={r.D:.3f}")
ax.set(xlabel=r"$\lambda$",ylabel="CCDF",title=f"{name}: direct MLE overlays"); ax.grid(True,which="both",alpha=.2); ax.legend(fontsize=7); fig.savefig(out/"powerlaw_ccdf_overlays.png",dpi=180); plt.show()

compare=[]
for method,r in (("weightwatcher_standard",first),("weightwatcher_clip_xmax",first_clip)):
    compare.append({"method":method,"trim_top_k":0 if method.endswith("standard") else fnum(r.get("num_fingers",np.nan)),"scope":"weightwatcher","support":"weightwatcher","alpha":fnum(r.get("alpha",np.nan)),"D":fnum(r.get("D",np.nan)),"xmin":fnum(r.get("xmin",np.nan)),"xmax":fnum(r.get("xmax",np.nan)),"n_tail":np.nan})
for _,r in primary[primary.trim_top_k.isin(candidates)].iterrows(): compare.append({"method":"powerlaw_MLE","trim_top_k":int(r.trim_top_k),"scope":r.scope,"support":r.support,"alpha":r.alpha,"D":r.D,"xmin":r.xmin,"xmax":r.xmax_model,"n_tail":int(r.n_tail)})
comparison=pd.DataFrame(compare); comparison.to_csv(out/"weightwatcher_vs_powerlaw_mle.csv",index=False); display(comparison)
print("saved diagnostics to",out)

## Interpretation

- If the parallel histogram bands are partitioned by exact occupancies and collapse to discrete levels in $\lambda\widehat\rho$, they are bin-occupancy ridges rather than evidence for two physical densities.
- The `full, unbounded, k=0` row tests a power law across the complete ESD. The `tail` rows show the MLE after `powerlaw` selects $x_{min}$; always inspect `n_tail` with $\alpha$ and KS distance.
- A credible finger breakpoint is a sharp KS improvement followed by relative stability of $\alpha$, $x_{min}$, and tail size. A merely monotonic improvement as observations are removed is not enough.
- The bounded-support fit is shown separately because imposing `xmax` changes the statistical model, not only the data subset.
